# 🚀 Multi-Pair Backtest Lab
Welcome to the interactive backtesting environment. This notebook allows you to test and optimize trading strategies with real-time visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Import modular components
from config import DATA_FOLDER, RESULT_FOLDER, TradingConfig
from indicators import calculate_atr, calculate_ichimoku, calculate_bollinger_bands, calculate_williams_r, calculate_stochastic
from strategies import identify_trend, generate_trend_signals, generate_mean_reversion_signals
from backtest import run_backtest
from analytics import analyze_performance
from visualization import plot_backtest_results
from strategy_finder import find_best_strategy, detect_timeframe

warnings.filterwarnings('ignore')
%matplotlib inline

## 1. Data Selection
Select a CSV file from the `candle_data` folder.

In [ ]:
def load_and_validate_data(filename):
    try:
        df = pd.read_csv(os.path.join(DATA_FOLDER, filename))
        df.columns = [col.lower().strip() for col in df.columns]
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date').reset_index(drop=True)
        if df[['date', 'open', 'high', 'low', 'close']].isnull().any().any():
            df = df.fillna(method='ffill')
        return df
    except Exception as e:
        print(f"Error loading data: {str(e)}")
        return None

csv_files = [f for f in os.listdir(DATA_FOLDER) if f.endswith('.csv')]
file_dropdown = widgets.Dropdown(options=csv_files, description='CSV File:')
display(file_dropdown)

## 2. Configuration & Strategy Selection
Configure your trading parameters and choose a strategy.

In [ ]:
strategy_dropdown = widgets.Dropdown(
    options=['TREND_FOLLOWING', 'MEAN_REVERSION', 'STRATEGY_FINDER'],
    value='TREND_FOLLOWING',
    description='Strategy:'
)

sl_input = widgets.FloatText(value=1.5, description='SL ATR Mult:')
tp_input = widgets.FloatText(value=2.0, description='TP ATR Mult:')
run_button = widgets.Button(description="🚀 Run Backtest", button_style='success')
output_area = widgets.Output()

display(widgets.VBox([strategy_dropdown, sl_input, tp_input, run_button, output_area]))

In [ ]:
def on_run_clicked(b):
    with output_area:
        clear_output()
        filename = file_dropdown.value
        strategy_type = strategy_dropdown.value
        custom_sl = sl_input.value
        custom_tp = tp_input.value
        
        basename = os.path.splitext(filename)[0].upper()
        selected_pair = basename.split('_')[0] if '_' in basename else basename
        
        print(f"Pair: {selected_pair} | Strategy: {strategy_type}")
        
        df = load_and_validate_data(filename)
        if df is None: return

        if strategy_type == 'STRATEGY_FINDER':
            print("Running Strategy Finder optimization...")
            best_strat = find_best_strategy(df, selected_pair, custom_sl, custom_tp)
            if best_strat == 'ML_STRATEGY':
                print("ML Strategy optimization complete.")
                return
            elif best_strat:
                strategy_type = best_strat
                print(f"Optimized Choice: {strategy_type}")
            else:
                return

        config = TradingConfig(selected_pair, strategy_type, sl_multiplier=custom_sl, tp_multiplier=custom_tp)
        
        # Calculate indicators
        df = calculate_atr(df, config.atr_period)
        if strategy_type == 'TREND_FOLLOWING':
            df = calculate_ichimoku(df, config.tenkan_period, config.kijun_period)
            df = calculate_stochastic(df, config.stoch_k_period, config.stoch_d_period, config.stoch_slowing)
            df = identify_trend(df)
            df = generate_trend_signals(df, config)
        else:
            df = calculate_bollinger_bands(df, config.bollinger_period, config.bollinger_std)
            df = calculate_williams_r(df, config.williams_period)
            df = generate_mean_reversion_signals(df, config)

        # Backtest
        trades, equity_curve = run_backtest(df, config)
        trades_df, metrics = analyze_performance(trades, equity_curve, config.initial_equity, selected_pair)
        
        # Visualization
        if trades_df is not None:
            display(HTML(f"<h3>Performance Summary for {selected_pair}</h3>"))
            metrics_df = pd.DataFrame(metrics.items(), columns=['Metric', 'Value'])
            display(metrics_df)
            
            # Inline plot
            plot_backtest_results(df, trades_df, equity_curve, config, metrics)

run_button.on_click(on_run_clicked)